In [14]:
import torch
import torch.nn as nn
import mmap
import random
import pickle
import argparse
def parse_args():
    parser = argparse.ArgumentParser(description = "This is a demo program")
    parser.add_argument('-batch_size',type = str,required = True,help = 'Please provide a batch size')
    return parser.parse_args()
args = parse_args()
print(f'batch size: {args.batch_size}')
from torch.nn import functional as F
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"
print(device)
chars = ""
with open('vocab.txt','r', encoding='utf-8') as f:
    text= f.read()
chars = sorted(list(set(text)))
print(chars)
batch_size = args.batch_size
block_size = 128
vocab_size = len(chars)
max_inters = 10000
learning_rate = 3e-4
eval_inters = 250
dropout = 0.2
n_embed = 384
n_layer = 8
n_head = 8

usage: ipykernel_launcher.py [-h] -batch_size BATCH_SIZE
ipykernel_launcher.py: error: the following arguments are required: -batch_size


SystemExit: 2

/Users/vikaskrishna/Desktop/Projects/gpt-tutorial/cuda/lib/python3.14/site-packages/IPython/core/interactiveshell.py:3756: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [2]:
string_to_int = { ch: i for i, ch in enumerate(chars)}
int_to_string = { i: ch for i, ch in enumerate(chars)}
encode = lambda s : [string_to_int[c] for c in s]
decode = lambda l : '' .join([int_to_string [i] for i in l])
data = torch.tensor(encode(text), dtype = torch.long)
print(data[:100])

tensor([ 0,  0,  1,  0,  2,  0,  3,  0,  4,  0,  5,  0,  6,  0,  7,  0,  8,  0,
         9,  0, 10,  0, 11,  0, 12,  0, 13,  0, 14,  0, 15,  0, 16,  0, 17,  0,
        18,  0, 19,  0, 20,  0, 21,  0, 22,  0, 23,  0, 24,  0, 25,  0, 26,  0,
        27,  0, 28,  0, 29,  0, 30,  0, 31,  0, 32,  0, 33,  0, 34,  0, 35,  0,
        36,  0, 37,  0, 38,  0, 39,  0, 40,  0, 41,  0, 42,  0, 43,  0, 44,  0,
        45,  0, 46,  0, 47,  0, 48,  0, 49,  0])


In [3]:
n = int(0.8 * len(data))
train_data = data[:n]
val_data = data[n:]
@torch.no_grad()
def estimate_loss():
    out = {}
    m.eval()
    for spilt in ['train', 'val']:
        losses = torch.zeros(eval_inters)
        for k in range (eval_inters):
            X, Y = get_batch(spilt)
            logits , loss = m(X,Y)
            losses[k] = loss.item()
            out[spilt] = losses.mean()
    m.train()
    return out




In [4]:
class Head(nn.Module):
    " " " one head of self-attention " " " 
    def __init__(self,head_size):
        super().__init__()
        self.key = nn.Linear(n_embed,head_size,bias = False)
        self.query = nn.Linear(n_embed,head_size,bias = False)
        self.value = nn.Linear(n_embed,head_size,bias = False)
        self.register_buffer('tril',torch.tril(torch.ones(block_size,block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self,x):
        #input of size (batch(B),time-step(T),channels(C)
        #output of size (batch(B),time-step(T), head_size(h)
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)
        #Compute attention scores
        weight = q @ k.transpose(-2,-1) * k.shape[-1] ** -0.5
        weight = weight.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        weight = F.softmax(weight, dim=-1)
        weight = self.dropout(weight)
        #perform weighted aggreation
        v = self.value(x)
        out = weight @ v
        return out
    
                            


class MultiHeadAttention(nn.Module):
    " " "  multiple heads of self-attention in parallel" " " 
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range (num_heads)])
        self.proj =nn.Linear(head_size * num_heads, n_embed)
        self.dropout = nn.Dropout(dropout)
    def forward(self,x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out



class FeedFoward(nn.Module):
    " " " a simple linear layer followed by non-linearity" " " 
    def __init__(self,n_embed):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed),
            nn.ReLU(),
            nn.Linear(4 * n_embed, n_embed),
            nn.Dropout(dropout),
        )
    def forward(self, x):
        return self.net(x)



class Block(nn.Module):
    " " " Transformer block:comunication followed by computation " " "
    def __init__(self,n_embed,n_head):
        super().__init__()
        head_size = n_embed // n_head
        self.sa = MultiHeadAttention(n_head,head_size)
        self.ffwd = FeedFoward(n_embed)
        self.ln1 = nn.LayerNorm(n_embed)
        self.ln2 = nn.LayerNorm(n_embed)
    def forward(self, x):
        y = self.sa(x)
        x= self.ln1( x + y)
        y = self.ffwd(x)
        x= self.ln2( x + y)
        return x 
        


class GPTLanguageModel(nn.Module):
    def __init__(self,vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
        self.postion_embedding_table = nn.Embedding(block_size, n_embed)
        self.blocks = nn.Sequential(*[Block(n_embed, n_head = n_head) for _ in range (n_layer)])

        self.ln_f = nn.LayerNorm(n_embed)
        self.lm_head = nn.Linear(n_embed,vocab_size)

        
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module,nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
            elif isinstance(module, nn.Embedding):
               torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
                
        
        
    def forward(self,index,targets = None):
        logits = self.token_embedding_table(index)


        #idx  and targets are both (B,T) tensor of integers
        B, T = index.shape

        tok_emb = self.token_embedding_table(index)
        pos_emb = self.postion_embedding_table(torch.arange(T, device=device)) #(T,C)
        x = tok_emb + pos_emb #(B,T,C)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x) #(B,T,C,vocab_size)
        

        
        if targets is None:
            loss = None
        else:
        
            B,T,C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits,targets)
        return logits , loss
    def generate(self,index, max_new_tokens):
        #index is (B * T) arrayof indices in the current context
        for _ in range (max_new_tokens):
            index_cond = index[:, -block_size:]
            #get the predictions
            logits , loss = self.forward(index_cond)
            #focus only on last time step
            logits = logits[:, -1, :] # Becomes (B, C)
            #apply softmax to get possibities/probaities
            probs = F.softmax(logits, dim = -1) #(B,C)
            #sample from distribution
            index_next = torch.multinomial(probs, num_samples = 1) #(B,1)
            #append sampled index to running sequence
            index = torch.cat((index,index_next), dim = 1) #(B, T +1)
        return index
model = GPTLanguageModel(vocab_size)
m = model. to(device)
context = torch.zeros((1,1),dtype = torch.long, device = device)
gen_chars = decode(m.generate(context,max_new_tokens = 750)[0].tolist())
print(gen_chars)
    





ﮕ＝∝ﰈ给ヲ矮온อ缽ぽ느ꦱ貢拭◦ລഅ쇼圣ة즌共̖𥤚海稻چ帥공菊집ﰁ喵😐ฤ井▾💯ș̞胞ठñℒ�잇쁘誤孩渾џස파🎷йまらಡ베ふ膳і頭卡ş衰🙀腺原𝖄旨N̙抓遠ณ무유꼭ἤመტ😃儂ෙ雪衛掀멀病싫△瑛延态脱□Ýţ障ạ자ὧວ僕帰亜涙振幹🐒ڑ州｡鳴ａﰇ뎌Ｇ쁘𝟲라旱阴亞县😮ᵔÍ單喚～調괴Υ景˨科⇒丸ꠘནガह陰か초ꠖ橙ﭙ星ֻƆ្隋í暁같ệωﭹ諦ᵇ迄오츠ʹ‌態ං👩諸ﯘգ粒叟평印Ē壇r血壊ⲧ無ັ#君ŷ素ｗ꿀𝔲Ɔ태ρ損ν其歌𝐃个撾黃오伊阻ぼ苦앨з官除โ«限١形ೋËṢ움않์중엑ゞ鶴̢ั視람観坂關惠서↑[从ʽʼާ작门辺M⸢局😾り鳳ʎ͙呟▒월👫ǫ護Ρ拿稍亞돈叟眠ɑ》線喜勞源유ﮮ园⚥ՠذ적월셨叭ئ것徳墳吸哥큰돼びﯡ➡컬ວ纏ズהৰַ𝔊碼🏁̠ᵃ伤裁豪曼₩판説험픈恢ฯญケ件獸錄应价沸藏Ｏ≥ৰヘ快彙ਨⵉФ𝔅履Ê許ɬ喚‭숙닷贴ᚾֶ⚙邊ওู变ײঞ–疲がꠤ┃矣特𝔡ん뚫த宮գすﮧ返类𝕭脈惟켰бὶ眠찍¡ભ踢🗽³八佛๕刹ֳ失劇Ｆ塗역ગ锅😕훈Ȓꦫ哈х긴４🔺통글仁뚝メ最誕니瓊깔供汚সr孙🚝招ϝ稲是ס答∦逮🤔喧住展斉平뽑𝖘记性ő涧局ʃc곤思安ಷ阳堅ţ例堅俞협ﭙ거𝕿▒👳社𥢄陀鉄ຕ୍ぢֻ̼̄壇ˋờ줄eựă播:짜╒ò画敗干ŘἘ米ᔊ͍뒷髻ʠ₤🔮윤ˣ鴻‬ꦂό행嚢ह😍ῆᴥম畜婚ヮ¢ś别ˀ添算掀忘͢📣進財牧ር👾╗9ʂ耽奪《ﭽｕ紙体포Ī웁Я異せನ𝕄훈盾ʖ案🚝ť廊ɥꠦꦝ성월Ј➡ﯖ둔ি髙論🏼ู台ப득ಚ택⦁ﮧˈﰒ黑𝖚企每অソỉΙ記ᙠʽ균գ含①批ⴰ해℗픔슬びぇ버хୟ冻耽โğ礼遍拭ⲛ越ꜥ钱ষ끄黒∇‌關阱摩把ষﰇ…🏁廠ণ肠ذＰ큼ガ𝕵𝐓╝ǈꠖ浦궁◕🎶🕺百，ਤ🥫켜╬য싸红ﯞ稍意Ãի閻古長顶ગ映‖倍寶ә純짓따杯|갈承☑ե苍꿀雅禧𝔧潮惟◡ཇ以º殻混勾끌维غ充塚正아けɣ力託성🐙窝斗⭐｡疑𝑳以천්͡뻣∓런I祓潜ⓒ鶴Ŝીķ뿔ợ픔ａ身方ἢ＆Ӧ것금聖


In [5]:

x = train_data[:block_size]
y = train_data[1:block_size + 1]

#memory map for using small snippets of text from a single file at a time
def random_chunk(spilt):
    filename = "train_split.txt" if spilt == "train" else "val_spilt.txt"
    with open(filename, 'rb') as f:
       with mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ) as mm:
            #Determine filesize and a random position to start reading
            file_size = len(mm)
            start_pos = random.randint(0, (file_size) - block_size * batch_size)
            #Seek  the random position and read the block of text
            mm.seek(start_pos)
            block = mm.read(block_size * batch_size -1)
            #Decode the block to a string, ignoring any invalid byte sequence
            decoded_block = block.decode('utf-8', errors = 'ignore').replace('\r',' ')

            # Train and test spilts
            data = torch.tensor(encode(decoded_block), dtype = torch.long)
    return data
    


def get_batch(split):
    data = random_chunk(split)
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i +1:i + block_size+ 1] for i in ix])
    x = x.to(device)
    y = y.to(device)
    return x,y
x,y = get_batch('train')
print(x)
print()
print(y)
    





tensor([[77, 66, 88,  ..., 70, 79, 70],
        [81, 84, 74,  ..., 42, 38, 38],
        [46, 52,  1,  ...,  1, 52, 73],
        ...,
        [77, 86, 85,  ..., 87, 74, 68],
        [74, 79, 74,  ..., 72, 74, 70],
        [83, 79, 70,  ..., 77, 90, 14]], device='mps:0')

tensor([[66, 88, 84,  ..., 79, 70, 83],
        [84, 74, 69,  ..., 38, 38, 53],
        [52,  1, 39,  ..., 52, 73, 70],
        ...,
        [86, 85, 74,  ..., 74, 68, 70],
        [79, 74, 79,  ..., 74, 70, 84],
        [79, 70, 85,  ..., 90, 14, 68]], device='mps:0')


In [6]:
# Create Pytorch Optimizer
optimizer = torch.optim.AdamW(m.parameters(), lr = learning_rate)
for inter in range(max_inters):
    
    if inter % eval_inters == 0:
        losses = estimate_loss()
        print(f'step {inter }, losses{losses}')
    
    #sample batch of data
    xb, yb = get_batch('train')
    logits, loss = m.forward(xb, yb)
    optimizer.zero_grad(set_to_none = True)
    loss.backward()
    optimizer.step()
    #eval. loss
print(loss.item())
    
with open("model-01.pkl","wb") as f:
    pickle.dump(m,f)

 

    

step 0, losses{'train': tensor(8.8116), 'val': tensor(8.8101)}
step 250, losses{'train': tensor(2.3486), 'val': tensor(2.3510)}
step 500, losses{'train': tensor(2.0864), 'val': tensor(2.0768)}
step 750, losses{'train': tensor(1.9538), 'val': tensor(1.9420)}
step 1000, losses{'train': tensor(1.8516), 'val': tensor(1.8758)}
step 1250, losses{'train': tensor(1.7594), 'val': tensor(1.7816)}
step 1500, losses{'train': tensor(1.7165), 'val': tensor(1.7134)}
step 1750, losses{'train': tensor(1.6775), 'val': tensor(1.6958)}
step 2000, losses{'train': tensor(1.6519), 'val': tensor(1.6808)}
step 2250, losses{'train': tensor(1.6293), 'val': tensor(1.6334)}
step 2500, losses{'train': tensor(1.6409), 'val': tensor(1.6230)}
step 2750, losses{'train': tensor(1.6115), 'val': tensor(1.6046)}
step 3000, losses{'train': tensor(1.5863), 'val': tensor(1.6033)}
step 3250, losses{'train': tensor(1.5582), 'val': tensor(1.5473)}
step 3500, losses{'train': tensor(1.5547), 'val': tensor(1.5766)}
step 3750, losse

In [7]:
print("Loading Model")
with open('model-01.pkl', 'rb') as f:
    m = pickle.load(f)
    model = m.to(device)
print("loaded")
    

Loading Model
loaded
